In [ ]:
import os
from pathlib import Path

# Ensure relative paths resolve from the project root when running from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

# Compare Viscosities Analysis
Updated to use Parquet files for better performance and memory efficiency

In [ ]:
import pandas as pd
from pathlib import Path
from klarity import parsing
from klarity import metrics
from klarity import viz

In [ ]:
import config
from klarity import io

In [ ]:
io.check_dataframes_stale()

## Load Data

**NEW:** We now load only the columns needed for this analysis.
This is much faster and uses less memory than loading all 43 columns!

### Alternative Loading Options

If you need to load data differently, here are some alternatives:

```python
# Option 1: Load only specific placements (faster, less memory)
df = parsing.load_all_data_parquet(
    PARQUET_DIR,
    placements=["placement_1", "placement_2"],  # Only these 2
    columns=['equivalent_diameter', 'mask_area', 'confidence']
)

# Option 2: Load one placement at a time (most memory efficient)
for placement in ["placement_1", "placement_2", ...]:
    df = parsing.load_placement_parquet(PARQUET_DIR, placement)
    # ... process ...
    del df

# Option 3: Use filters (only loads matching rows)
df = parsing.load_filtered_parquet(
    PARQUET_DIR,
    filters=[("confidence", ">", 0.8)],
    columns=['equivalent_diameter', 'bubble_volume_mm3']
)
```

## Compute Bubble-Level Metrics

This adds/updates volume and surface area calculations

## Save Bubble-Level Data (Optional)

Save for later use without reprocessing

In [ ]:
bubble_level_df = pd.read_pickle(config.BUBBLE_LEVEL_PKL)

## Compute Frame-Level Metrics

Aggregate bubble data to frame (image) level

## Create Visualizations

Generate comparison plots across xanthan concentrations

In [ ]:
# Configuration for plots
placements = [
    "placement_1",
    "placement_2",
    "placement_3",
    "placement_4",
    "placement_5",
    "placement_6",
]

print("Creating viscosity comparison plots...")
viz.plot_all_xanthan_grids(
    bubble_level_df=bubble_level_df,
    placements=placements,
    value_col="equivalent_diameter_mm",
    bins=40,
    outdir=config.VISC_COMPARISON_DIR,
    fname_prefix="visc_compare_settings",
    xmax_percentile=99.8,
)

print("\nOK Plots saved to visc_comparison/ directory")

## Summary Statistics (Optional)

In [ ]:
# Quick summary by placement and xanthan concentration
summary = (
    bubble_level_df.reset_index()
    .groupby(["placement", "reactor_setting"])["equivalent_diameter_mm"]
    .agg(["count", "mean", "median", "std"])
)

print("\nSummary Statistics:")
print(summary.head(20))

# Save summary
summary.to_csv("summary_statistics.csv")
print("\nOK Summary saved to summary_statistics.csv")

## Performance Notes

**With Parquet files:**
- Data loading: ~10-30 seconds (vs 2-5 minutes with CSV)
- Memory usage: ~1-3 GB (vs 90 GB with CSV)
- Only loads columns you need (43 columns available, but we only load ~10)

**Tips for even better performance:**
1. Load only needed columns (as done above)
2. Filter during loading with `load_filtered_parquet()`
3. Process placements one at a time if memory is tight
4. Save intermediate results (bubble_level_df, frame_level_df) as pickles